# 04 — QLoRA ile Instruction Fine-Tuning

**İlan maddesi:** *"Transformer tabanlı büyük dil modelleri üzerinde eğitim çalışmaları
yürütmek"*, *"LoRA, QLoRA, PEFT veya instruction tuning teknikleriyle fine-tuning."*

## Neden QLoRA?
- **Full fine-tuning**: Tüm parametreleri günceller. Milyarlarca parametre için Colab'ın
  ücretsiz T4 GPU'suna (~15GB VRAM) sığmaz.
- **LoRA**: Temel modeli dondurur, her katmana küçük "adaptör" matrisleri ekler; sadece
  bunları eğitir (parametrelerin <%1'i).
- **QLoRA**: LoRA'nın üstüne, temel modeli 4-bit'e (NF4) sıkıştırarak bellek ihtiyacını
  daha da azaltır — böylece 3B-7B modeller tek bir T4'e sığar.

Bu notebook, 01. notebook'ta ürettiğiniz `sft_dataset.jsonl` ile devam eder.

In [ ]:
# Bu hücre HER notebook'ta ayrı ayrı çalıştırılmalı: Colab'da her sekme/notebook
# genellikle kendi çalışma zamanını (VM) alır, yani /content her seferinde sıfırdanmış
# gibi başlar. Bu hücre kendi kendini onaran bir kurulum yapar:
#   1) Proje klasörü zaten varsa (aynı çalışma zamanında önceki hücre/notebook
#      tarafından kurulmuşsa) hiçbir şey yapmadan devam eder.
#   2) Yoksa Google Drive'ı mount edip, DRIVE_ZIP_PATH'teki zip'i /content'e açar
#      (zip'in içinde 'baykar-nlp-hazirlik/' klasörü kök olarak yer almalı).
#   3) Drive'da zip de yoksa, kendi GitHub reponuzu klonlamanız için bir uyarı basar.
import os, sys

PROJECT_DIR = "/content/baykar-nlp-hazirlik"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/baykar-nlp-hazirlik.zip"

if not os.path.exists(PROJECT_DIR):
    try:
        from google.colab import drive
        # drive.mount() zaten mount edilmişse anında geri döner (idempotent);
        # os.path.exists("/content/drive") ile "mount edilmiş mi" kontrol etmek
        # güvenilmez çünkü klasör, başarısız/yarım bir mount denemesinden sonra
        # bile var olabilir. Bu yüzden koşulsuz çağırıyoruz.
        drive.mount("/content/drive", force_remount=True)
        if os.path.exists(DRIVE_ZIP_PATH):
            import shutil
            shutil.unpack_archive(DRIVE_ZIP_PATH, "/content")
        else:
            print(f"UYARI: {DRIVE_ZIP_PATH} bulunamadı. Zip'i Drive'ınızın köküne "
                  "yükleyin ya da kendi reponuzu klonlayın: "
                  f"!git clone <repo-url> {PROJECT_DIR}")
    except ImportError:
        pass  # Colab dışında (yerelde) çalışıyorsanız bu adım gerekmez.

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)

sys.path.insert(0, PROJECT_DIR)


In [ ]:
import torch
print("CUDA kullanılabilir mi:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("UYARI: GPU bulunamadı. Runtime > Değiştir çalışma zamanı türü > T4 GPU seçin.")


## Hiperparametreler

`src/config.py` içindeki `QLoraConfig`'i inceleyin: `lora_r`, `lora_alpha`, `target_modules` gibi değerler burada tanımlı.

In [ ]:
from src.config import QLORA_CONFIG
print(QLORA_CONFIG)


## Eğitim

Bu hücre GPU'da birkaç dakika ile birkaç saat arasında sürebilir (veri seti boyutuna bağlı).

In [ ]:
from src.finetune.qlora_train import train

adapter_path = train()
print("LoRA adaptörü kaydedildi ->", adapter_path)


## Hızlı doğrulama

Eğitilen adaptörle RAG pipeline'ını tekrar çalıştırıp, temel modelle karşılaştırın (bkz. 07. notebook A/B test).

In [ ]:
from src.rag.rag_pipeline import answer

result = answer("Baykar hangi tür insansız hava araçları üretir?", model_path=adapter_path)
print(result["answer"])
